# 11 · Governance vs Hydra
*Hydra tries to read what it should not.*

Hydra has an agent inside the building. Fury wants every analyst to see the briefings, only a few people to see real codenames, and nobody below clearance 4 to see the top-clearance agents.
In Unity Catalog that is four tools: privileges (`GRANT`), column masks, row filters and tags. A fifth, lineage, tells you afterwards where data came from and who touched it.

| Concept | Where you meet it |
|---|---|
| Privileges and least privilege by layer | the next section, Step 1 |
| Column mask | Step 3 |
| Row filter | Step 4 |
| Tags | Step 5 |
| Lineage and audit (system tables) | Step 6 |

Deep dive: [docs/11-governance-vs-hydra.md](../../docs/11-governance-vs-hydra.md)

**Needs:** chapter 04 (`shield.silver.agents`: 50 agents, 51 after its MERGE); for Step 6 the gold table `shield.gold.daily_threat_briefing` (chapter 06 or 10). Run it on serverless compute or a SQL warehouse (row filters and column masks do not work on every compute type).
You must own `shield.silver.agents` (or have `MANAGE` and `SELECT` on it). The group `fury` should be an **account-level group** (account console → User management → Groups). Until it exists the policies still run: everybody is simply "not Fury".
The notebook measures whether *you* are in `fury` and adapts its **Expected** values, so it passes either way. It changes the table's policies and tags and removes all of them in the clean-up.

## Access control in Unity Catalog

**The idea:** several independent layers decide what a person sees. Privileges decide whether they may query a table at all. Inside a table, a column mask changes the *values* they see and a row filter hides *rows*. Tags describe the data. Lineage records where it flows.
Masks and filters are features of the table, not of a query tool, so they hold in notebooks, SQL and BI alike. They add restrictions on top of privileges and never grant access themselves.

```
 metastore ─▶ catalog shield ─▶ schema gold ─▶ table daily_threat_briefing
              USE CATALOG       USE SCHEMA      SELECT          (privileges are inherited downwards;
                                                                 reading a table needs all three)

 query on shield.silver.agents
   1. privileges     may this user query the table at all?           (Step 1)
   2. row filter     which rows survive?                             (Step 4)
   3. column mask    what value does each column show?               (Step 3)
```

| Tool | Question it answers | Mechanism |
|---|---|---|
| Privilege | may this group touch it? | `GRANT USE CATALOG / USE SCHEMA / SELECT` to an account-level **group** |
| Column mask | what does this person see in the column? | a SQL function bound to the column (`SET MASK`) |
| Row filter | which rows does this person see? | a SQL function bound to the table (`SET ROW FILTER`) |
| Tag | what kind of data is it? | key-value labels (`classification = secret`) |
| Lineage | where did it come from, who used it? | recorded automatically, queryable from system tables |

**How this notebook implements it**

* *Least privilege by layer*: analysts get `shield.gold` only, engineers keep bronze and silver. Step 1 looks at what is granted today and shows the statements; it runs none of them, because it cannot know your group names.
* Both functions use `is_account_group_member('fury')`. `shield.gold.mask_codename` returns the codename for Fury and `*** CLASSIFIED ***` for everyone else; `shield.gold.clearance_filter` lets Fury see every row and everyone else only clearance 3 and below.
* Step 2 records how `shield.silver.agents` looks without policies, so the later steps compare with real numbers.
* Step 5 tags `codename` with `classification = secret` and `pii = true`. Step 6 reads lineage.

**Why it helps:** one membership change onboards or offboards a person; the rules live next to the data and cannot be forgotten in a dashboard.

**Good to know:**
* Learn describes table-level row filters and column masks as the manual way. **Attribute-based access control (ABAC)** applies tag-driven policies across a catalog or schema and is the recommended approach for many tables. It was **Public Preview** in the November 2025 release notes; check its current status. This chapter does not cover it.
* The table owner can change or remove the policies. Whether *your* account is exempt from them is not something this repo has verified: the check tables tell you what you actually see.
* The function parameter types must match the column types, otherwise values are cast silently or to `NULL`. The notebook reads the type of `clearance_level` from the table.
* `CREATE OR REPLACE TABLE` keeps an existing row filter and masks (Learn), so leftovers from an aborted run would follow the table into the next chapter. The clean-up removes them.

**Not covered yet:** ABAC, dynamic views, real groups and grants (the doc has the statements), the audit system table `system.access.audit`.

## Setup

In [ ]:
import contextlib

from pyspark.sql import functions as F

AGENTS = "shield.silver.agents"


def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected.
    An expected value of None means there is no single right answer: compare by eye (shown as –)."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    mark = lambda got, want: "–" if want is None else ("✅" if got == want else "❌")
    rows = [(what, fmt(got), "–" if want is None else fmt(want), mark(got, want)) for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


assert spark.catalog.tableExists(AGENTS), "Run notebook 04 first. Missing table: " + AGENTS
in_fury = spark.sql("SELECT is_account_group_member('fury')").first()[0]
show_checks([("you are a member of the account group fury", in_fury, None)])

## 1 · Privileges

### Step 1 · What is granted on gold today?
**Goal:** see the privileges on the schema analysts will use and that you can list its tables. The notebook does not grant anything, because it does not know your groups. With a group `analysts` you would run:

| Statement | Effect |
|---|---|
| ``GRANT USE CATALOG ON CATALOG shield TO `analysts` `` | may enter the catalog |
| ``GRANT USE SCHEMA, SELECT ON SCHEMA shield.gold TO `analysts` `` | may read every table in gold, today and later |
| (nothing on `bronze` and `silver`) | engineers keep those |

**Expected:** `SHOW GRANTS` lists whatever grants exist; on a fresh catalog it may be empty, because ownership is not a grant. You can list the gold tables of chapter 05, 06 or 10 (at least one).

In [ ]:
display(spark.sql("SHOW GRANTS ON SCHEMA shield.gold"))
gold_tables = spark.sql("SHOW TABLES IN shield.gold").collect()
show_checks([("tables you can list in shield.gold", len(gold_tables) >= 1, True)])

## 2 · Look first, then protect

### Step 2 · The agents without policies
**Goal:** record the baseline: how many agents, with which clearance, before any mask or filter.
The cell first drops a mask or filter that an earlier aborted run may have left behind (there is nothing to drop on a clean table, so errors are ignored).

**Expected** (chapter 04 as written; 50 agents, 51 after its MERGE):

| check | result |
|---|---|
| agents | 50 or 51 |
| agents with clearance 1 to 3 | 29 |
| agents with clearance 4 or 5 | 21 or 22 |

The rows shown are the first five agents by id, with real codenames, for example `A0001  Nomad-13  3` (before chapter 04's MERGE).

In [ ]:
for statement in [
    f"ALTER TABLE {AGENTS} DROP ROW FILTER",
    f"ALTER TABLE {AGENTS} ALTER COLUMN codename DROP MASK",
]:
    with contextlib.suppress(Exception):  # nothing to drop on a clean table
        spark.sql(statement)

agents_baseline = spark.table(AGENTS)
agents_total = agents_baseline.count()
clearance_type = dict(agents_baseline.dtypes)["clearance_level"]
agents_by_clearance = {row["clearance_level"]: row["count"] for row in agents_baseline.groupBy("clearance_level").count().collect()}
agents_low = sum(n for level, n in agents_by_clearance.items() if level <= 3)
agents_high = agents_total - agents_low

show_checks([
    ("agents", agents_total in (50, 51), True),
    ("agents with clearance 1 to 3", agents_low, 29),
    ("agents with clearance 4 or 5", agents_high in (21, 22), True),
])
display(agents_baseline.select("agent_id", "codename", "clearance_level").orderBy("agent_id").limit(5))

## 3 · Column mask

### Step 3 · Only Fury sees real codenames
**Goal:** bind a function to `codename` so the value depends on who asks.

**Expected:**
* If you are **not** in `fury`: every codename reads `*** CLASSIFIED ***` (all agents), for example `A0001  *** CLASSIFIED ***`.
* If you are in `fury`: no codename is masked.

If you are not in `fury` and still see real codenames, you may be exempt as the table owner (not verified): ask a colleague to run the same query, or follow Try it 1 in the doc.

In [ ]:
spark.sql(
    """
    CREATE OR REPLACE FUNCTION shield.gold.mask_codename(codename STRING)
    RETURN CASE WHEN is_account_group_member('fury') THEN codename ELSE '*** CLASSIFIED ***' END
    """
)
spark.sql(f"ALTER TABLE {AGENTS} ALTER COLUMN codename SET MASK shield.gold.mask_codename")

agents_masked = spark.table(AGENTS)
show_checks([
    ("agents showing '*** CLASSIFIED ***'", agents_masked.filter("codename = '*** CLASSIFIED ***'").count(), 0 if in_fury else agents_total),
])
display(agents_masked.select("agent_id", "codename", "clearance_level").orderBy("agent_id").limit(5))

## 4 · Row filter

### Step 4 · You only see agents at or below your clearance
**Goal:** bind a function to the table so rows disappear for people who should not see them. Here only `fury` members see clearance 4 and 5.
The function's parameter has the same type as the table column (read in Step 2).

**Expected:**
* If you are **not** in `fury`: only clearance 1 to 3 remain, 29 agents, and the highest clearance you see is 3.
* If you are in `fury`: all agents stay.

What you will see for a non-member:

| clearance_level | agents |
|---|---|
| 1 | 9 (10 after chapter 04's MERGE) |
| 2 | 9 |
| 3 | 11 (10 after the MERGE) |

In [ ]:
spark.sql(
    f"""
    CREATE OR REPLACE FUNCTION shield.gold.clearance_filter(clearance {clearance_type})
    RETURN is_account_group_member('fury') OR clearance <= 3
    """
)
spark.sql(f"ALTER TABLE {AGENTS} SET ROW FILTER shield.gold.clearance_filter ON (clearance_level)")

agents_filtered = spark.table(AGENTS)
visible_by_clearance = agents_filtered.groupBy("clearance_level").count().orderBy("clearance_level")
display(visible_by_clearance)
show_checks([
    ("agents you can see", agents_filtered.count(), agents_total if in_fury else agents_low),
    ("highest clearance you can see", agents_filtered.agg(F.max("clearance_level")).first()[0], max(agents_by_clearance) if in_fury else 3),
])

## 5 · Tags

### Step 5 · Make sensitive data discoverable
**Goal:** label the column so people (and, later, policies) can find sensitive data without reading the code. You need the `APPLY TAG` privilege, which the owner has.

**Expected:** `shield.information_schema.column_tags` lists two tags for `silver.agents.codename`:

| column_name | tag_name | tag_value |
|---|---|---|
| codename | classification | secret |
| codename | pii | true |

In [ ]:
spark.sql(f"ALTER TABLE {AGENTS} ALTER COLUMN codename SET TAGS ('classification' = 'secret', 'pii' = 'true')")

codename_tags = spark.sql(
    """
    SELECT column_name, tag_name, tag_value
    FROM shield.information_schema.column_tags
    WHERE schema_name = 'silver' AND table_name = 'agents'
    ORDER BY tag_name
    """
)
display(codename_tags)
show_checks([
    ("tags on silver.agents", sorted((r["tag_name"], r["tag_value"]) for r in codename_tags.collect()), [("classification", "secret"), ("pii", "true")]),
])

## 6 · Lineage and audit

### Step 6 · Where does the briefing come from?
**Goal:** answer "if `silver.locations` is wrong, which gold tables are affected?" from recorded lineage, not from memory.
Catalog Explorer shows the same as a graph (see below). The system table needs a metastore admin to enable the `access` schema first; without it the cell prints a message and moves on.

**Expected (not verified):** the sources of `shield.gold.daily_threat_briefing` include `shield.silver.incident_reports` and `shield.silver.locations`. A ❌ can also mean that lineage was not recorded yet: system tables can lag behind, and only lineage the platform could infer is recorded.

In [ ]:
try:
    briefing_sources = spark.sql(
        """
        SELECT DISTINCT source_table_full_name, target_table_full_name
        FROM system.access.table_lineage
        WHERE target_table_full_name = 'shield.gold.daily_threat_briefing'
        ORDER BY source_table_full_name
        """
    )
    display(briefing_sources)
    sources = {row["source_table_full_name"] for row in briefing_sources.collect()}
    show_checks([
        ("silver.incident_reports is a source", "shield.silver.incident_reports" in sources, True),
        ("silver.locations is a source", "shield.silver.locations" in sources, True),
    ])
except Exception as error:  # noqa: BLE001
    print("System tables not enabled yet:", type(error).__name__)

## 👀 In the workspace

* **Catalog** → `shield` → `silver` → `agents`: the **Overview** tab shows the *Row filter* and the *Column mask* (fx icons) and the tags on `codename`. Remove them there to see what the clean-up does in SQL.
* **Catalog** → `shield` → `gold` → `daily_threat_briefing` → **Lineage** → **See lineage graph**: upstream tables, notebooks and jobs.
* The audit trail is in `system.access.audit` once the `access` schema is enabled.

## 🧹 Clean up

Later chapters must see real data, and `CREATE OR REPLACE TABLE` would carry a leftover mask or filter into them. The cell removes everything this chapter created, in this order: the filter and the mask first, then the functions (dropping a function that a table still uses leaves the table unreadable), then the tags.
Nothing was granted in this notebook, so there is nothing to revoke.

In [ ]:
for statement in [  # the order matters: policies before the functions they use
    f"ALTER TABLE {AGENTS} DROP ROW FILTER",
    f"ALTER TABLE {AGENTS} ALTER COLUMN codename DROP MASK",
    "DROP FUNCTION IF EXISTS shield.gold.clearance_filter",
    "DROP FUNCTION IF EXISTS shield.gold.mask_codename",
    f"ALTER TABLE {AGENTS} ALTER COLUMN codename UNSET TAGS ('classification', 'pii')",
]:
    try:
        spark.sql(statement)
    except Exception as error:  # noqa: BLE001
        print("skipped (probably never created):", statement, "-", type(error).__name__)

agents_after = spark.table(AGENTS)
tags_left = spark.sql("SELECT * FROM shield.information_schema.column_tags WHERE schema_name = 'silver' AND table_name = 'agents'").count()
show_checks([
    ("agents visible again", agents_after.count(), agents_total),
    ("agents still masked", agents_after.filter("codename = '*** CLASSIFIED ***'").count(), 0),
    ("tags left on silver.agents", tags_left, 0),
])

## 🎯 Recap

* Privileges decide who may query a table; a column mask changes what they see in a column, a row filter changes which rows they see. Masks and filters belong to the table, so every tool obeys them.
* Both policies here depend on one question, `is_account_group_member('fury')`, and the notebook adapted its checks to your answer.
* Tags make sensitive columns findable, and lineage shows what feeds a gold table. Everything created here is removed again.

Mini-projects: [docs/11-governance-vs-hydra.md](../../docs/11-governance-vs-hydra.md#try-it-yourself)

> 💡 Next: chapter 12 makes the queries behind those briefings cheaper and faster.